In [10]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

In [11]:
# Helper functions
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    model = "claude-sonnet-4-6",
    tools=[],
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "tools": tools,
        "max_tokens": 2048,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


In [ ]:
import json

with open("./sample.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
messages = []
first_message = [
    # document block
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        },
        "title": "sampl.pdf",
        "citations": {"enabled": True},
        "cache_control": {
            "type": "ephemeral"
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "Summarize file with one sentence",
    },
]

add_user_message(messages, first_message)
response = chat(messages)
add_assistant_message(messages, response)
response

Message(id='msg_01KnW3mDhr4R5hxvCMheBXUL', container=None, content=[TextBlock(citations=None, text='This document provides a step-by-step guide on ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='Getting an API Key from VoyageAI\r\nStep 1: Sign up to VoyageAI\r\n1. ', document_index=0, document_title='sample.pdf', end_page_number=2, file_id=None, start_page_number=1, type='page_location'), CitationPageLocation(cited_text='Create an account\r\nStep 2: Create an API key\r\n1. ', document_index=0, document_title='sample.pdf', end_page_number=2, file_id=None, start_page_number=1, type='page_location'), CitationPageLocation(cited_text='Step 3: Add the key to your “.env” file\r\n1. ', document_index=0, document_title='sample.pdf', end_page_number=4, file_id=None, start_page_number=3, type='page_location')], text='getting an API key from VoyageAI, covering how to sign up, create an API key, and add it to your ".env" file.', type='text')], model='claude-sonnet-4-6', role=